## Rocchio-SVM demo

In [16]:
from textprep.vectorizer import to_remove_symbols, to_lower
import pandas as pd

from utils.download import *
from pulearn.pubase import *


download_from_gdrive("deceptive-opinion.csv")

documents = pd.read_csv("../data/deceptive-opinion.csv")
documents["deceptive"] = documents["deceptive"].apply(lambda x: 0 if x == "truthful" else 1)
documents["label"] = documents["deceptive"].copy()
documents = documents.drop(columns=["hotel", "source", "polarity", "deceptive"])
documents["text"] = documents["text"].apply(lambda row: to_remove_symbols(to_lower(row)))
documents.head()

[~_o] Dataset found locally --> Aborting download


,text,label
0,we stayed for a one night getaway with family ...,0
1,triple a rate with upgrade to view room was le...,0
2,this comes a little late as im finally catchin...,0
3,the omni chicago really delivers on all fronts...,0
4,i asked for a high floor away from the elevato...,0


Isolate all deceptive documents and extract a partial test set of 160 instances

In [17]:
deceptive_docs = documents.loc[(documents["label"] == 1)]
deceptive_docs.reset_index(drop=True, inplace=True)
indices = extract_sample(deceptive_docs["label"], ratio=0.2, value=1)
deceptive_set = deceptive_docs.loc[indices]
deceptive_set

,text,label
412,i would never stay here again for a hotel that...,1
651,i stayed at the monacochicago back in april i ...,1
671,i decided to spend the extra money booking a r...,1
757,my experience at the amalfi hotel in chicago w...,1
484,after considering several hotels in the area m...,1
...,...,...
242,great hotel went for the weekend with my wife ...,1
541,i recently had the misfortune of staying at th...,1
346,my wife and i stayed here for a weekend and we...,1
333,i went here with the family including our dog ...,1


Isolate all non-deceptive documents and extract a partial test set of another 160 instances

In [18]:
non_deceptive_docs = documents.loc[(documents["label"] == 0)]
non_deceptive_docs.reset_index(drop=True, inplace=True)
indices = extract_sample(non_deceptive_docs["label"], ratio=0.2, value=0)
non_deceptive_set = non_deceptive_docs.loc[indices]
non_deceptive_set

,text,label
733,the palmer house has a beautiful lobby with a ...,0
266,great place great room great location even tho...,0
174,my girlfriends and i stayed 4 nights at the ta...,0
312,we stayed here from nov 30 to dec 2 and had a ...,0
781,often visit chicago but this was my first stay...,0
...,...,...
96,the best service the staff here was incredible...,0
704,october 3rd 2007 my wife and i stayed at the s...,0
60,great hotel in heart of chicago for business o...,0
53,i stay at this hotel 2 times a year on busines...,0


Merge above sets to create a unified test set

In [19]:
test_set = pd.concat([deceptive_set, non_deceptive_set], ignore_index=True)
test_set

,text,label
0,i would never stay here again for a hotel that...,1
1,i stayed at the monacochicago back in april i ...,1
2,i decided to spend the extra money booking a r...,1
3,my experience at the amalfi hotel in chicago w...,1
4,after considering several hotels in the area m...,1
...,...,...
315,the best service the staff here was incredible...,0
316,october 3rd 2007 my wife and i stayed at the s...,0
317,great hotel in heart of chicago for business o...,0
318,i stay at this hotel 2 times a year on busines...,0


Accordingly extract a training set and turn it into a PU dataset

In [20]:
train_set = documents.loc[~(documents["text"].isin(test_set["text"]))]
train_set = train_set.reset_index(drop=True)

indices = extract_sample(train_set["label"], ratio=0.37, value=1)
train_set["pu-label"] = 0
train_set.loc[indices, "pu-label"] = 1

train_set["pu-label"].value_counts()

0    1042
1     237
Name: pu-label, dtype: int64

Create TF-IDF vectors

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer


X_train = train_set["text"]
y_train = train_set["pu-label"]

X_test = test_set["text"]
y_test = test_set["label"]

tfidf_input = pd.concat([X_train, X_test])

vectorizer = TfidfVectorizer()

vectors = vectorizer.fit_transform(tfidf_input)
feature_names = vectorizer.get_feature_names_out()
dense = vectors.todense().tolist()

X = pd.DataFrame(dense, columns=feature_names)

train_rows = X_train.shape[0]
test_rows = X_test.shape[0]

X_train = X.iloc[:train_rows]
X_test = X.iloc[-test_rows:]
X_test = X_test.reset_index(drop=True)

Growing classifier

In [22]:
from pulearn.iterative.growing import GrowingClassifier
from sklearn.metrics import f1_score


roc_svm = GrowingClassifier("Rocchio", "SVC", params1={"metric": "manhattan"}, params2={"kernel": "linear"})
roc_svm.fit(X_train, y_train)
predictions = roc_svm.predict(X_test)

print("\nF1-score = {:.2f}".format(f1_score(y_test, predictions, average="macro") * 100))

Iteration: 0
[[  0 246]
 [  1 122]]
Iteration: 1
[[ 0 82]
 [ 1 40]]
Iteration: 2
[[ 0 33]
 [ 1  7]]
Iteration: 3
[[0 5]
 [1 2]]
Iteration: 4


Pruning Classifier -- currently contains a bug which results in an endless loop

Reason: internal 2nd estimator **sometimes** makes only negative predictions & results in while condition never exiting

In [ ]:
from pulearn.iterative.pruning import PruningClassifier


roc_svm = PruningClassifier("Rocchio", "SVC", params1={"metric": "manhattan"}, params2={"kernel": "linear"})
roc_svm.fit(X_train, y_train)
predictions = roc_svm.predict(X_test)

print("\nF1-score = {:.2f}".format(f1_score(y_test, predictions, average="macro") * 100))